# Memory-Split PoC — Facts in Context (no DB retrieval)

**Question.** At a fixed parameter budget, does a model that *offloads facts to context* — never memorizing them — reason better than a dense twin that stores them in weights?

One corpus, two arms, **one toggle**. Every fact appears in a `Context:` block in the training text. Only the loss mask on the fact **value** differs:
- **DENSE** — loss ON everywhere → memorizes `(subject, relation) → value` into weights.
- **SPLIT** — loss OFF on the value in context → never memorizes it; must READ it from context. Facts offloaded, capacity freed.

Facts are real Wikidata triples (PopQA). **Three open-book tasks:** fact-QA (single-hop), reason-over-facts (yes/no comparison — answer ≠ any single fact), and pure reasoning (`puremath`: compute e.g. `a mod b`; the operation's *definition* is the relevant fact given in context — a closer starting point, not the answer).

**Fair evaluation, four conditions** (closed-book AND +context for both arms, so 'does context help?' is a within-arm comparison): DENSE @ closed-book · DENSE + context · SPLIT @ closed-book · SPLIT + context. Fact-QA graded string-match **and** by an LLM judge (`--judge`).

**Scale:** team's `d160m` (~162M) by default; `--model d360m` (~356M, A100). GPU runtime; paste a TrueFoundry token (used only to grade answers).

**Note:** fresh experiment — checkpoints live under `runs/<model>_incontext/`, separate from any earlier DB run.

## 1. Get the code (syncs to the latest branch state)

In [ ]:
import os
REPO_URL = "https://github.com/sidvenkatayogi/Memory-Split.git"
BRANCH = "poc/optimal-retriever"
if os.path.basename(os.getcwd()) != "Memory-Split":
    if not os.path.isdir("Memory-Split"):
        !git clone --branch {BRANCH} --single-branch {REPO_URL}
    %cd Memory-Split
!git fetch -q origin {BRANCH} && git reset --hard -q FETCH_HEAD
!git log --oneline -1

In [ ]:
# Colab preinstalls torch; install only the missing runtime deps (never touch torch).
!pip install -q tiktoken openai
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
                    '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Persist to Google Drive (checkpoints + results survive a disconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['POC_PERSIST_DIR'] = '/content/drive/MyDrive/memory_split_poc'
os.makedirs(os.environ['POC_PERSIST_DIR'], exist_ok=True)
print('persist dir ->', os.environ['POC_PERSIST_DIR'])

## 3. TrueFoundry credentials (used only to grade fact-QA answers)

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("TrueFoundry token: ")
os.environ["OPENAI_BASE_URL"] = "https://tfy.promptlens.trilogy.com/v1"
os.environ["POC_GPT_MODEL"] = "openai-group/gpt-5.6-sol"

import sys; sys.path.insert(0, ".")
from evals.gpt_oracle import GatewayClient
print("gateway smoke ->", GatewayClient().smoke())  # e.g. 'Paris'

## 4. Build the corpus (offline; facts in context; split masks the values)

In [ ]:
!python scripts/poc_run.py --stage build

## 5. Train the matched twins (DENSE then SPLIT)
Same corpus, model, budget, init — only the value loss-mask differs. Reuses a finished checkpoint on Drive if present (`--fresh` to retrain; raise `--steps` to train longer).

In [ ]:
!python scripts/poc_run.py --stage train --device auto --model d160m --steps 4000 --ckpt-minutes 5

## 6. Evaluate + report
Four conditions (DENSE/SPLIT × closed-book/+context) across all three tasks — fact-QA (string-match + LLM judge), reason-over-facts, and pure reasoning.

In [ ]:
!python scripts/poc_run.py --stage eval --device auto --judge
!python scripts/poc_run.py --stage report

In [ ]:
import json, os
from IPython.display import Image, display
persist = os.environ.get('POC_PERSIST_DIR', 'data/poc')
print(json.dumps(json.load(open(f'{persist}/poc_results.json')), indent=2))
display(Image(f'{persist}/poc_figure.png'))